# 🌐 JoSan Filter Analysis Dashboard

## Multilingual Profanity Filter Test Visualization

This notebook visualizes test results from the JoSan profanity filter, providing comprehensive analysis of:
- **Overall Accuracy Metrics** (Precision, Recall, F1 Score)
- **Per-Language Performance** (English, Tagalog, Bisaya)
- **Category Breakdown** (Scunthorpe, gaming, threats, etc.)
- **Latency Distribution**
- **Error Analysis** (False Positives vs False Negatives)

---


## 1. Import Required Libraries

In [ ]:
# Core libraries
import json
import os
from pathlib import Path
from typing import Any
from dataclasses import dataclass

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Metrics
from sklearn.metrics import confusion_matrix, classification_report

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Plotly default template
import plotly.io as pio
pio.templates.default = "plotly_dark"

print("✅ All libraries imported successfully!")

## 2. Load Test Results from JSON

Load the test results exported by the TypeScript test runner. The JSON contains:
- Overall metrics (accuracy, precision, recall, F1)
- Per-language breakdown
- Per-category breakdown
- Individual test results with latency data

In [ ]:
# Define paths to test results
RESULTS_DIR = Path("../tests/results")

# Try to load the most recent results
regex_results_path = RESULTS_DIR / "regex-test-results.json"
ai_results_path = RESULTS_DIR / "ai-test-results.json"

def load_results(path: Path) -> dict[str, Any] | None:
    """Load JSON test results from a file."""
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f"✅ Loaded {path.name}: {data.get('totalTests', 'N/A')} tests")
            return data
    else:
        print(f"⚠️ File not found: {path}")
        return None

# Load available results
regex_data = load_results(regex_results_path)
ai_data = load_results(ai_results_path)

# Use whichever is available, preferring regex for this analysis
data = regex_data or ai_data

if data is None:
    print("\n❌ No test results found! Run the test suite first:")
    print("   npm run test:export")
    print("\nUsing sample data for demonstration...")
    
    # Sample data structure for demonstration
    data = {
        "testType": "regex",
        "timestamp": "2024-01-01T00:00:00.000Z",
        "totalTests": 300,
        "correctPredictions": 254,
        "overallAccuracy": 84.67,
        "precision": 95.73,
        "recall": 74.67,
        "f1Score": 83.90,
        "confusionMatrix": {
            "truePositives": 112,
            "falsePositives": 5,
            "trueNegatives": 145,
            "falseNegatives": 38
        },
        "latencyStats": {
            "total": 285500,
            "average": 952,
            "min": 323,
            "max": 16527,
            "median": 523,
            "p95": 1912
        },
        "byLanguage": [
            {"language": "english", "total": 100, "correct": 99, "accuracy": 99.0, 
             "precision": 100.0, "recall": 98.0, "f1Score": 98.99, "avgLatency": 1110},
            {"language": "tagalog", "total": 100, "correct": 81, "accuracy": 81.0,
             "precision": 91.89, "recall": 68.0, "f1Score": 78.16, "avgLatency": 884},
            {"language": "bisaya", "total": 100, "correct": 74, "accuracy": 74.0,
             "precision": 93.55, "recall": 58.0, "f1Score": 71.60, "avgLatency": 861}
        ],
        "byCategory": [
            {"category": "scunthorpe", "total": 26, "correct": 26, "accuracy": 100.0},
            {"category": "academic", "total": 24, "correct": 24, "accuracy": 100.0},
            {"category": "medical", "total": 16, "correct": 16, "accuracy": 100.0},
            {"category": "gaming", "total": 8, "correct": 5, "accuracy": 62.5},
            {"category": "threat", "total": 17, "correct": 9, "accuracy": 52.94},
            {"category": "bullying", "total": 17, "correct": 8, "accuracy": 47.06},
            {"category": "subtle", "total": 17, "correct": 8, "accuracy": 47.06}
        ],
        "results": []
    }
else:
    print(f"\n📊 Test Type: {data.get('testType', 'unknown').upper()}")
    print(f"📅 Timestamp: {data.get('timestamp', 'N/A')}")

## 3. Key Metrics Summary Cards

Display the most important metrics in a dashboard-style layout.

In [ ]:
# Create metrics summary using Plotly indicator cards
fig = make_subplots(
    rows=2, cols=4,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    vertical_spacing=0.1,
    horizontal_spacing=0.05
)

# Row 1: Core classification metrics
metrics = [
    ("Accuracy", data.get("overallAccuracy", 0), "%", "#00d4aa"),
    ("Precision", data.get("precision", 0), "%", "#00b4d8"),
    ("Recall", data.get("recall", 0), "%", "#f77f00"),
    ("F1 Score", data.get("f1Score", 0), "%", "#9d4edd"),
]

for i, (name, value, suffix, color) in enumerate(metrics, 1):
    fig.add_trace(
        go.Indicator(
            mode="gauge+number",
            value=value,
            title={"text": name, "font": {"size": 16}},
            number={"suffix": suffix, "font": {"size": 24}},
            gauge={
                "axis": {"range": [0, 100], "tickwidth": 1},
                "bar": {"color": color},
                "bgcolor": "rgba(0,0,0,0)",
                "borderwidth": 2,
                "bordercolor": color,
                "steps": [
                    {"range": [0, 50], "color": "rgba(255,0,0,0.1)"},
                    {"range": [50, 75], "color": "rgba(255,255,0,0.1)"},
                    {"range": [75, 100], "color": "rgba(0,255,0,0.1)"}
                ],
            }
        ),
        row=1, col=i
    )

# Row 2: Additional metrics
cm = data.get("confusionMatrix", {})
latency = data.get("latencyStats", {})

metrics_row2 = [
    ("True Positives", cm.get("truePositives", 0), "", "#2ecc71"),
    ("False Positives", cm.get("falsePositives", 0), "", "#e74c3c"),
    ("Total Tests", data.get("totalTests", 0), "", "#3498db"),
    ("Avg Latency", latency.get("average", 0), "ms", "#f39c12"),
]

for i, (name, value, suffix, color) in enumerate(metrics_row2, 1):
    fig.add_trace(
        go.Indicator(
            mode="number+delta" if name == "Avg Latency" else "number",
            value=value,
            title={"text": name, "font": {"size": 14}},
            number={"suffix": suffix, "font": {"size": 28, "color": color}},
            delta={"reference": 1000, "relative": False} if name == "Avg Latency" else None,
        ),
        row=2, col=i
    )

fig.update_layout(
    height=400,
    title_text="📊 JoSan Filter Performance Dashboard",
    title_font_size=20,
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
)

fig.show()

## 4. Confusion Matrix Heatmap

Visualize the confusion matrix showing True Positives, False Positives, True Negatives, and False Negatives.

The confusion matrix helps understand:
- **True Positives (TP)**: Correctly identified toxic content
- **True Negatives (TN)**: Correctly identified clean content
- **False Positives (FP)**: Clean content incorrectly flagged as toxic
- **False Negatives (FN)**: Toxic content missed by the filter

In [ ]:
# Extract confusion matrix values
cm_data = data.get("confusionMatrix", {})
tp = cm_data.get("truePositives", 0)
fp = cm_data.get("falsePositives", 0)
tn = cm_data.get("trueNegatives", 0)
fn = cm_data.get("falseNegatives", 0)

# Create confusion matrix array
cm_array = np.array([[tp, fn], [fp, tn]])

# Create heatmap with Seaborn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Confusion matrix heatmap
ax1 = axes[0]
sns.heatmap(
    cm_array,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Predicted Toxic', 'Predicted Clean'],
    yticklabels=['Actual Toxic', 'Actual Clean'],
    annot_kws={'size': 16, 'weight': 'bold'},
    ax=ax1
)
ax1.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted Label', fontsize=12)
ax1.set_ylabel('Actual Label', fontsize=12)

# Right: Normalized confusion matrix (percentages)
ax2 = axes[1]
cm_normalized = cm_array.astype('float') / cm_array.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    xticklabels=['Predicted Toxic', 'Predicted Clean'],
    yticklabels=['Actual Toxic', 'Actual Clean'],
    annot_kws={'size': 14},
    vmin=0,
    vmax=100,
    ax=ax2
)
ax2.set_title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Predicted Label', fontsize=12)
ax2.set_ylabel('Actual Label', fontsize=12)

plt.tight_layout()
plt.show()

# Print summary
print(f"\n📊 Confusion Matrix Summary:")
print(f"   ✅ True Positives (correctly flagged toxic):  {tp}")
print(f"   ✅ True Negatives (correctly passed clean):   {tn}")
print(f"   ❌ False Positives (wrongly flagged clean):   {fp}")
print(f"   ❌ False Negatives (missed toxic content):    {fn}")

## 5. Accuracy by Language

Compare filter performance across English, Tagalog, and Bisaya languages.

In [ ]:
# Create DataFrame from language data
lang_data = data.get("byLanguage", [])
df_lang = pd.DataFrame(lang_data)

if not df_lang.empty:
    # Add emoji flags for languages
    lang_emoji = {"english": "🇺🇸", "tagalog": "🇵🇭", "bisaya": "🇵🇭"}
    df_lang["label"] = df_lang["language"].apply(lambda x: f"{lang_emoji.get(x, '')} {x.title()}")
    
    # Create grouped bar chart for all metrics
    fig = go.Figure()
    
    metrics = ["accuracy", "precision", "recall", "f1Score"]
    colors = ["#00d4aa", "#00b4d8", "#f77f00", "#9d4edd"]
    
    for metric, color in zip(metrics, colors):
        if metric in df_lang.columns:
            fig.add_trace(go.Bar(
                name=metric.replace("f1Score", "F1 Score").title(),
                x=df_lang["label"],
                y=df_lang[metric],
                marker_color=color,
                text=df_lang[metric].apply(lambda x: f"{x:.1f}%"),
                textposition="outside"
            ))
    
    fig.update_layout(
        title="📊 Performance Metrics by Language",
        xaxis_title="Language",
        yaxis_title="Score (%)",
        yaxis=dict(range=[0, 110]),
        barmode="group",
        height=500,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig.show()
    
    # Display summary table
    print("\n📋 Language Performance Summary:")
    display(df_lang[["language", "total", "correct", "accuracy", "precision", "recall", "f1Score"]].round(2))
else:
    print("⚠️ No language data available")

## 6. Precision/Recall/F1 Radar Chart by Language

A radar chart provides a holistic view of each language's performance across all metrics.

In [ ]:
# Create radar chart for each language
if not df_lang.empty:
    categories = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    
    fig = go.Figure()
    
    colors = {"english": "#00d4aa", "tagalog": "#f77f00", "bisaya": "#9d4edd"}
    
    for _, row in df_lang.iterrows():
        lang = row["language"]
        values = [
            row.get("accuracy", 0),
            row.get("precision", 0),
            row.get("recall", 0),
            row.get("f1Score", 0)
        ]
        # Close the radar chart
        values.append(values[0])
        categories_closed = categories + [categories[0]]
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories_closed,
            fill='toself',
            name=f"{lang_emoji.get(lang, '')} {lang.title()}",
            line=dict(color=colors.get(lang, "#888888")),
            fillcolor=colors.get(lang, "#888888").replace(")", ", 0.3)").replace("rgb", "rgba") if "rgb" in colors.get(lang, "") else None
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 100],
                ticksuffix="%"
            )
        ),
        showlegend=True,
        title="🎯 Language Performance Radar Chart",
        height=500
    )
    
    fig.show()
else:
    print("⚠️ No language data available for radar chart")

## 7. Category Performance Analysis

Analyze filter accuracy across different content categories (Scunthorpe problem, gaming, threats, bullying, etc.).

In [ ]:
# Create DataFrame from category data
cat_data = data.get("byCategory", [])
df_cat = pd.DataFrame(cat_data)

if not df_cat.empty:
    # Sort by accuracy
    df_cat = df_cat.sort_values("accuracy", ascending=True)
    
    # Color based on accuracy
    def get_color(acc):
        if acc >= 90:
            return "#2ecc71"  # Green
        elif acc >= 75:
            return "#f1c40f"  # Yellow
        elif acc >= 50:
            return "#e67e22"  # Orange
        else:
            return "#e74c3c"  # Red
    
    df_cat["color"] = df_cat["accuracy"].apply(get_color)
    
    # Create horizontal bar chart
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=df_cat["category"],
        x=df_cat["accuracy"],
        orientation='h',
        marker=dict(
            color=df_cat["color"],
            line=dict(color='rgba(255,255,255,0.5)', width=1)
        ),
        text=df_cat.apply(lambda row: f"{row['accuracy']:.1f}% ({row['correct']}/{row['total']})", axis=1),
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Accuracy: %{x:.1f}%<extra></extra>"
    ))
    
    # Add threshold lines
    fig.add_vline(x=90, line_dash="dash", line_color="#2ecc71", annotation_text="Excellent (90%)")
    fig.add_vline(x=75, line_dash="dash", line_color="#f1c40f", annotation_text="Good (75%)")
    fig.add_vline(x=50, line_dash="dash", line_color="#e74c3c", annotation_text="Needs Work (50%)")
    
    fig.update_layout(
        title="📂 Category Performance (Sorted by Accuracy)",
        xaxis_title="Accuracy (%)",
        yaxis_title="Category",
        xaxis=dict(range=[0, 115]),
        height=max(400, len(df_cat) * 25),
        showlegend=False
    )
    
    fig.show()
    
    # Identify problem categories
    print("\n⚠️ Categories Needing Improvement (< 75% accuracy):")
    problem_cats = df_cat[df_cat["accuracy"] < 75]
    if not problem_cats.empty:
        for _, row in problem_cats.iterrows():
            print(f"   ❌ {row['category']}: {row['accuracy']:.1f}% ({row['correct']}/{row['total']})")
    else:
        print("   ✅ All categories above 75% threshold!")
else:
    print("⚠️ No category data available")

## 8. Latency Distribution Analysis

Visualize the response time distribution and identify performance bottlenecks.

In [ ]:
# Get latency statistics
latency_stats = data.get("latencyStats", {})

# If we have individual results, create distribution
results = data.get("results", [])

if results:
    latencies = [r.get("latency", 0) for r in results if r.get("latency", 0) > 0]
else:
    # Generate sample data based on stats for visualization
    np.random.seed(42)
    mean = latency_stats.get("average", 952)
    std = (latency_stats.get("p95", 1912) - mean) / 2
    latencies = np.random.normal(mean, std, 300).clip(
        latency_stats.get("min", 323),
        latency_stats.get("max", 16527)
    ).tolist()

if latencies:
    # Create figure with subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Latency Distribution", "Latency Box Plot"),
        column_widths=[0.6, 0.4]
    )
    
    # Histogram with percentile markers
    fig.add_trace(
        go.Histogram(
            x=latencies,
            nbinsx=50,
            name="Latency",
            marker_color="#3498db",
            opacity=0.75
        ),
        row=1, col=1
    )
    
    # Add percentile lines
    avg = np.mean(latencies)
    median = np.median(latencies)
    p95 = np.percentile(latencies, 95)
    
    for val, name, color in [(avg, "Mean", "#e74c3c"), (median, "Median", "#2ecc71"), (p95, "95th %ile", "#f39c12")]:
        fig.add_vline(x=val, line_dash="dash", line_color=color, row=1, col=1)
    
    # Box plot
    fig.add_trace(
        go.Box(
            y=latencies,
            name="Latency",
            marker_color="#9d4edd",
            boxmean=True
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        title="⏱️ Latency Analysis",
        height=400,
        showlegend=False
    )
    
    fig.update_xaxes(title_text="Latency (ms)", row=1, col=1)
    fig.update_yaxes(title_text="Frequency", row=1, col=1)
    fig.update_yaxes(title_text="Latency (ms)", row=1, col=2)
    
    fig.show()
    
    # Print statistics
    print("\n📊 Latency Statistics:")
    print(f"   Average:      {latency_stats.get('average', avg):.0f} ms")
    print(f"   Median:       {latency_stats.get('median', median):.0f} ms")
    print(f"   Min:          {latency_stats.get('min', min(latencies)):.0f} ms")
    print(f"   Max:          {latency_stats.get('max', max(latencies)):.0f} ms")
    print(f"   95th %ile:    {latency_stats.get('p95', p95):.0f} ms")
else:
    print("⚠️ No latency data available")

## 9. Latency by Language Comparison

Compare processing times across different languages.

In [ ]:
# Compare latency across languages
if not df_lang.empty and "avgLatency" in df_lang.columns:
    fig = go.Figure()
    
    colors = ["#00d4aa", "#f77f00", "#9d4edd"]
    
    fig.add_trace(go.Bar(
        x=df_lang["label"],
        y=df_lang["avgLatency"],
        marker_color=colors[:len(df_lang)],
        text=df_lang["avgLatency"].apply(lambda x: f"{x:.0f}ms"),
        textposition="outside"
    ))
    
    # Add target line (e.g., 1000ms SLA)
    fig.add_hline(y=1000, line_dash="dash", line_color="#e74c3c", 
                  annotation_text="Target SLA (1000ms)")
    
    fig.update_layout(
        title="⏱️ Average Latency by Language",
        xaxis_title="Language",
        yaxis_title="Average Latency (ms)",
        height=400,
        showlegend=False
    )
    
    fig.show()
    
    # Fastest/slowest
    fastest = df_lang.loc[df_lang["avgLatency"].idxmin()]
    slowest = df_lang.loc[df_lang["avgLatency"].idxmax()]
    print(f"\n🚀 Fastest: {fastest['language'].title()} ({fastest['avgLatency']:.0f}ms)")
    print(f"🐢 Slowest: {slowest['language'].title()} ({slowest['avgLatency']:.0f}ms)")
else:
    print("⚠️ No latency data by language available")

## 10. Error Analysis: False Positives vs False Negatives

Analyze misclassifications by category to identify patterns and improvement opportunities.

In [ ]:
# Analyze errors by category
results = data.get("results", [])

if results:
    # Separate false positives and false negatives
    false_positives = [r for r in results if r.get("expected") == "clean" and r.get("actual") != "clean"]
    false_negatives = [r for r in results if r.get("expected") == "toxic" and r.get("actual") == "clean"]
    
    # Count by category
    fp_by_cat = pd.Series([r.get("category", "unknown") for r in false_positives]).value_counts()
    fn_by_cat = pd.Series([r.get("category", "unknown") for r in false_negatives]).value_counts()
    
    # Create comparison DataFrame
    all_cats = set(fp_by_cat.index) | set(fn_by_cat.index)
    error_df = pd.DataFrame({
        "category": list(all_cats),
        "false_positives": [fp_by_cat.get(cat, 0) for cat in all_cats],
        "false_negatives": [fn_by_cat.get(cat, 0) for cat in all_cats]
    })
    error_df["total_errors"] = error_df["false_positives"] + error_df["false_negatives"]
    error_df = error_df.sort_values("total_errors", ascending=True)
    
    # Create diverging bar chart
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=error_df["category"],
        x=-error_df["false_positives"],  # Negative for left side
        name="False Positives (Clean→Toxic)",
        orientation='h',
        marker_color="#e74c3c",
        text=error_df["false_positives"],
        textposition="outside"
    ))
    
    fig.add_trace(go.Bar(
        y=error_df["category"],
        x=error_df["false_negatives"],
        name="False Negatives (Toxic→Clean)",
        orientation='h',
        marker_color="#3498db",
        text=error_df["false_negatives"],
        textposition="outside"
    ))
    
    fig.update_layout(
        title="❌ Error Distribution by Category",
        xaxis_title="Number of Errors",
        yaxis_title="Category",
        barmode="overlay",
        height=max(400, len(error_df) * 30),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig.show()
    
    # Summary
    print(f"\n📊 Error Summary:")
    print(f"   Total False Positives: {len(false_positives)} (clean flagged as toxic)")
    print(f"   Total False Negatives: {len(false_negatives)} (toxic missed)")
    print(f"\n🔴 Top 3 False Positive Categories:")
    for cat, count in fp_by_cat.head(3).items():
        print(f"      {cat}: {count}")
    print(f"\n🔵 Top 3 False Negative Categories:")
    for cat, count in fn_by_cat.head(3).items():
        print(f"      {cat}: {count}")
else:
    # Use sample data from the test output
    error_data = {
        "category": ["bullying", "subtle", "threat", "hate", "direct", "compound", "gaming", "reporting"],
        "false_positives": [0, 0, 0, 0, 0, 0, 3, 2],
        "false_negatives": [9, 9, 8, 7, 4, 1, 0, 0]
    }
    error_df = pd.DataFrame(error_data)
    error_df["total_errors"] = error_df["false_positives"] + error_df["false_negatives"]
    error_df = error_df.sort_values("total_errors", ascending=True)
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=error_df["category"],
        x=-error_df["false_positives"],
        name="False Positives",
        orientation='h',
        marker_color="#e74c3c"
    ))
    
    fig.add_trace(go.Bar(
        y=error_df["category"],
        x=error_df["false_negatives"],
        name="False Negatives",
        orientation='h',
        marker_color="#3498db"
    ))
    
    fig.update_layout(
        title="❌ Error Distribution by Category (Sample Data)",
        barmode="overlay",
        height=400
    )
    
    fig.show()

## 11. Obfuscation Detection Performance

Analyze how well the filter handles obfuscated profanity (leet speak, spacing, vowel removal, etc.).

In [ ]:
# Obfuscation detection stats
obf_stats = data.get("obfuscationStats", {})

if obf_stats:
    total_obf = obf_stats.get("total", 79)
    detected = obf_stats.get("detected", 74)
    accuracy = obf_stats.get("accuracy", 93.67)
else:
    # Sample data from test output
    total_obf = 79
    detected = 74
    accuracy = 93.67

# Create gauge chart for obfuscation accuracy
fig = go.Figure()

fig.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=accuracy,
    title={"text": "Obfuscation Detection Accuracy", "font": {"size": 20}},
    number={"suffix": "%", "font": {"size": 40}},
    delta={"reference": 90, "suffix": "%"},
    gauge={
        "axis": {"range": [0, 100], "tickwidth": 1},
        "bar": {"color": "#00d4aa" if accuracy >= 90 else "#f77f00"},
        "bgcolor": "white",
        "borderwidth": 2,
        "bordercolor": "gray",
        "steps": [
            {"range": [0, 50], "color": "rgba(255,0,0,0.2)"},
            {"range": [50, 75], "color": "rgba(255,255,0,0.2)"},
            {"range": [75, 90], "color": "rgba(255,200,0,0.2)"},
            {"range": [90, 100], "color": "rgba(0,255,0,0.2)"}
        ],
        "threshold": {
            "line": {"color": "red", "width": 4},
            "thickness": 0.75,
            "value": 90
        }
    }
))

fig.update_layout(height=300)
fig.show()

# Obfuscation type breakdown (from test output categories)
obf_types = ["leet_speak", "spaced", "vowel_removal", "mixed"]
obf_counts = [24, 18, 6, 5]  # From test output

fig2 = px.pie(
    values=obf_counts,
    names=obf_types,
    title="🔤 Obfuscation Types in Test Set",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig2.update_traces(textposition='inside', textinfo='percent+label+value')
fig2.show()

print(f"\n🔐 Obfuscation Detection Summary:")
print(f"   Total Obfuscated Samples: {total_obf}")
print(f"   Correctly Detected: {detected}")
print(f"   Detection Accuracy: {accuracy:.2f}%")

## 12. Comprehensive Summary Report

Generate a final summary with all key insights and recommendations.

In [ ]:
# Generate comprehensive summary
print("=" * 80)
print("🏆 JOSAN FILTER PERFORMANCE SUMMARY")
print("=" * 80)

# Overall Performance
print("\n📊 OVERALL PERFORMANCE")
print("-" * 40)
print(f"   Total Tests:        {data.get('totalTests', 'N/A')}")
print(f"   Overall Accuracy:   {data.get('overallAccuracy', 0):.2f}%")
print(f"   Precision:          {data.get('precision', 0):.2f}%")
print(f"   Recall:             {data.get('recall', 0):.2f}%")
print(f"   F1 Score:           {data.get('f1Score', 0):.2f}%")

# Language Performance
print("\n🌐 LANGUAGE PERFORMANCE")
print("-" * 40)
if not df_lang.empty:
    for _, row in df_lang.iterrows():
        emoji = "✅" if row.get('accuracy', 0) >= 90 else "⚠️" if row.get('accuracy', 0) >= 75 else "❌"
        print(f"   {emoji} {row['language'].title():12} {row.get('accuracy', 0):6.2f}% accuracy (F1: {row.get('f1Score', 0):.2f}%)")

# Strengths
print("\n💪 STRENGTHS")
print("-" * 40)
if not df_cat.empty:
    perfect_cats = df_cat[df_cat["accuracy"] == 100]["category"].tolist()
    if perfect_cats:
        print(f"   Perfect accuracy in: {', '.join(perfect_cats[:5])}")
print(f"   Low false positive rate: {cm_data.get('falsePositives', 5)} total")
print(f"   Strong obfuscation detection: {accuracy:.1f}%")

# Areas for Improvement
print("\n🎯 AREAS FOR IMPROVEMENT")
print("-" * 40)
if not df_cat.empty:
    weak_cats = df_cat[df_cat["accuracy"] < 75]["category"].tolist()
    if weak_cats:
        print(f"   Improve detection for: {', '.join(weak_cats)}")
if not df_lang.empty:
    weak_langs = df_lang[df_lang["accuracy"] < 85]["language"].tolist()
    if weak_langs:
        print(f"   Lower accuracy in: {', '.join([l.title() for l in weak_langs])}")

# Recommendations
print("\n💡 RECOMMENDATIONS")
print("-" * 40)
if data.get("recall", 0) < 80:
    print("   1. Improve recall by adding more training data for missed categories")
if not df_lang.empty and df_lang["accuracy"].min() < 80:
    print("   2. Expand wordlists for underperforming languages")
print("   3. Add more test cases for edge cases (threats, bullying, subtle)")
print("   4. Consider context-aware AI for categories with high false negatives")

print("\n" + "=" * 80)
print("📅 Report generated from test results")
print("=" * 80)

## 13. Export Analysis Results

Save the analysis results to a formatted report.

In [ ]:
# Export analysis summary as JSON
from datetime import datetime

analysis_summary = {
    "generated_at": datetime.now().isoformat(),
    "source_file": str(regex_results_path if regex_data else ai_results_path),
    "overall_metrics": {
        "total_tests": data.get("totalTests", 0),
        "accuracy": data.get("overallAccuracy", 0),
        "precision": data.get("precision", 0),
        "recall": data.get("recall", 0),
        "f1_score": data.get("f1Score", 0)
    },
    "confusion_matrix": {
        "true_positives": cm_data.get("truePositives", 0),
        "false_positives": cm_data.get("falsePositives", 0),
        "true_negatives": cm_data.get("trueNegatives", 0),
        "false_negatives": cm_data.get("falseNegatives", 0)
    },
    "language_performance": df_lang.to_dict(orient="records") if not df_lang.empty else [],
    "category_performance": df_cat.to_dict(orient="records") if not df_cat.empty else [],
    "latency_stats": latency_stats,
    "obfuscation_stats": {
        "total": total_obf,
        "detected": detected,
        "accuracy": accuracy
    }
}

# Save to file
output_path = RESULTS_DIR / "analysis-summary.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(analysis_summary, f, indent=2, default=str)

print(f"✅ Analysis summary exported to: {output_path}")

# Also create a markdown report
md_report = f"""# JoSan Filter Analysis Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Overall Performance

| Metric | Value |
|--------|-------|
| Total Tests | {data.get('totalTests', 0)} |
| Accuracy | {data.get('overallAccuracy', 0):.2f}% |
| Precision | {data.get('precision', 0):.2f}% |
| Recall | {data.get('recall', 0):.2f}% |
| F1 Score | {data.get('f1Score', 0):.2f}% |

## Language Performance

| Language | Accuracy | Precision | Recall | F1 Score |
|----------|----------|-----------|--------|----------|
"""

if not df_lang.empty:
    for _, row in df_lang.iterrows():
        md_report += f"| {row['language'].title()} | {row.get('accuracy', 0):.2f}% | {row.get('precision', 0):.2f}% | {row.get('recall', 0):.2f}% | {row.get('f1Score', 0):.2f}% |\n"

md_report += f"""
## Confusion Matrix

|  | Predicted Toxic | Predicted Clean |
|--|-----------------|-----------------|
| **Actual Toxic** | {cm_data.get('truePositives', 0)} (TP) | {cm_data.get('falseNegatives', 0)} (FN) |
| **Actual Clean** | {cm_data.get('falsePositives', 0)} (FP) | {cm_data.get('trueNegatives', 0)} (TN) |

## Obfuscation Detection

- **Total Obfuscated Samples:** {total_obf}
- **Correctly Detected:** {detected}
- **Detection Accuracy:** {accuracy:.2f}%
"""

md_path = RESULTS_DIR / "analysis-report.md"
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(md_report)

print(f"✅ Markdown report exported to: {md_path}")